In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

In [4]:
from sebastian.clients.deepseek.credentials import DeepSeekApiKey

c = DeepSeekApiKey(api_key="YOUR_DEEPSEEK_API_KEY")

In [6]:
from cloud.helper.secrets import get_secret, SecretKeys


credentials = get_secret(SecretKeys.DeepSeekCredentials)

In [9]:
from pydantic import BaseModel, Field


class Joke(BaseModel):
    genre: str = Field(description="The genre of the joke, e.g., 'programming', 'knock-knock', etc.")
    joke: str = Field(description="The content of the joke.")


In [21]:
import requests

url = "https://api.deepseek.com/chat/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {credentials.api_key}",
}

payload = {
    "model": "deepseek-v4-flash",
    "messages": [
        {"role": "system", "content": f"Return only valid JSON with the schema {Joke.model_json_schema()}"},
        {"role": "user", "content": "Tell me a joke about programmers. Set genre to programming."},
    ],
    "thinking": {"type": "enabled"},
    "reasoning_effort": "high",
    "response_format": {"type": "json_object"},
    "stream": False,
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
print(response.status_code)
print(response.json())

200
{'id': '0ac91d59-e9dd-4d50-8033-5990cb979476', 'object': 'chat.completion', 'created': 1786786079, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{"genre": "programming", "joke": "Why do programmers prefer dark mode? Because light attracts bugs."}', 'reasoning_content': "We need to output JSON with genre and joke. The joke about programmers. Let's create one. Ensure JSON valid."}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 219, 'completion_tokens': 50, 'total_tokens': 269, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 219}, 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e'}


In [22]:
response_data = response.json()
response_text = response_data["choices"][0]["message"]["content"]
joke = Joke.model_validate_json(response_text)
joke

Joke(genre='programming', joke='Why do programmers prefer dark mode? Because light attracts bugs.')